State: 
0 = Sạch, 
1 = Bụi, 
2 = Robot, 
3 = Chướng ngại vật

In [417]:
import tkinter as tk
from tkinter import ttk, scrolledtext
import random
import time
from collections import deque
import heapq
import math

In [418]:
class Node:
    name_counter = 0

    @classmethod
    def reset_counter(cls):
        cls.name_counter = 0

    @classmethod
    def get_next_name(cls):
        # Tự động sinh tên Node: A, B, C... Z, AA, AB...
        n = cls.name_counter
        cls.name_counter += 1
        name = ""
        while True:
            name = chr(n % 26 + 65) + name
            n = n // 26 - 1
            if n < 0: break
        return name

    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.name = Node.get_next_name()
        # Cost bằng cost của cha + 1 (Root có cost = 0)
        self.cost = 0 if parent is None else parent.cost + 1

In [419]:
class BeliefNode:
    name_counter = 0

    @classmethod
    def reset_counter(cls):
        cls.name_counter = 0

    @classmethod
    def get_next_name(cls):
        n = cls.name_counter
        cls.name_counter += 1
        name = ""
        while True:
            name = chr(n % 26 + 65) + name
            n = n // 26 - 1
            if n < 0: break
        return "BS_" + name

    def __init__(self, states, parent=None, action=None):
        self.states = states  # List chứa các ma trận trạng thái (ở đây là 2)
        self.parent = parent
        self.action = action
        self.name = BeliefNode.get_next_name()
        self.cost = 0 if parent is None else parent.cost + 1

# Trả về chuỗi đại diện cho BS để kiểm tra trùng lặp trong danh sách Reached/Frontier
def bs_to_str(states):
    res = []
    for state in states:
        state_str = "[" + ",".join("[" + ",".join(str(val) for val in row) + "]" for row in state) + "]"
        res.append(state_str)
    return " | ".join(res)

# Kiểm tra xem BS đã đạt Goal chưa (Tất cả các trạng thái con đều phải sạch bụi)
def bs_goal_test(states):
    for state in states:
        if not goal_test(state):
            return False
    return True

# Hàm apply action an toàn: Khi bị mù, robot có thể thực hiện hành động đâm vào tường.
# Nếu đâm vào tường hoặc ra ngoài map, robot đứng im.
def safe_apply_action(state, action):
    new_state = copy_matrix(state)
    pos = find_robot(new_state)
    if not pos: return new_state
    
    x, y = pos
    nx, ny = x, y
    if action == "UP": nx -= 1
    elif action == "DOWN": nx += 1
    elif action == "LEFT": ny -= 1
    elif action == "RIGHT": ny += 1

    rows, cols = len(new_state), len(new_state[0])
    if 0 <= nx < rows and 0 <= ny < cols and new_state[nx][ny] != 3:
        new_state[x][y] = 0
        new_state[nx][ny] = 2
        
    return new_state

# Dừng trạng thái đã đạt Goal, tiếp tục chạy trạng thái chưa đạt
def bs_result(states, action):
    new_states = []
    for s in states:
        if goal_test(s):
            new_states.append(copy_matrix(s)) # Đã Goal thì giữ nguyên không di chuyển nữa
        else:
            new_states.append(safe_apply_action(s, action))
    return new_states

def bs_child_node(node, action):
    new_states = bs_result(node.states, action)
    return BeliefNode(new_states, node, action)

In [420]:
def copy_matrix(matrix):
    return [row[:] for row in matrix]

In [421]:
def find_robot(matrix):
    rows = len(matrix)
    cols = len(matrix[0])
    
    for i in range(rows):
        for j in range(cols):
            if matrix[i][j] == 2:
                return [i, j]
            
    return None

In [422]:
def goal_test(state):
    for row in state:
        if 1 in row:
            return False
    return True

In [423]:
def actions(state):
    x, y = find_robot(state)
    move = []
    
    rows = len(state)
    cols = len(state[0])
    
    if x > 0 and state[x-1][y] != 3:
        move.append("UP")
    if x < rows - 1 and state[x+1][y] != 3:
        move.append("DOWN")
    if y > 0 and state[x][y-1] != 3:
        move.append("LEFT")
    if y < cols - 1 and state[x][y+1] != 3:
        move.append("RIGHT")
        
    return move

In [424]:
def result(state, action):
    new_state = copy_matrix(state)

    x, y = find_robot(new_state)
    new_state[x][y] = 0

    nx = x
    ny = y

    if action == "UP":
        nx -= 1
    elif action == "DOWN":
        nx += 1
    elif action == "LEFT":
        ny -= 1
    elif action == "RIGHT":
        ny += 1

    new_state[nx][ny] = 2
    return new_state

In [425]:
def child_node(node, action):
    new_state = result(node.state, action)
    return Node(new_state, node, action)

In [426]:
def solution(node):
    path = []

    while node.parent is not None:
        path.append(node.action)
        node = node.parent
    path.reverse()

    return path

In [427]:
def move_robot(matrix, action):
    x, y = find_robot(matrix)

    matrix[x][y] = 0

    if action == "UP":
        x -= 1
    elif action == "DOWN":
        x += 1
    elif action == "LEFT":
        y -= 1
    elif action == "RIGHT":
        y += 1

    matrix[x][y] = 2

In [428]:
def count_dust(matrix):
    count = 0

    for row in matrix:
        count += row.count(1)

    return count

In [429]:
def breadth_first_search_1(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = []

    while len(frontier) > 0:
        node = frontier.popleft() # Lấy node ở đầu frontier ra (Queue: FIFO)
        reached.append(node.state)
        
        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        gui.log_step(node, frontier, reached, "Reached") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
     
    return None

In [430]:
def breadth_first_search_2(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    explored = []

    while len(frontier) > 0:
        node = frontier.popleft() # Lấy node ở đầu frontier ra (Queue: FIFO)
        explored.append(node.state)

        for action in actions(node.state):
            child = child_node(node, action)

            in_explored = False
            for s in explored:
                if s == child.state:
                    in_explored = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_explored and not in_frontier:
                if goal_test(child.state):
                    return solution(child)

                frontier.append(child)
        
        gui.log_step(node, frontier, explored, "Explored") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
                
    return None

In [431]:
def depth_first_search_1(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = []

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        gui.log_step(node, frontier, reached, "Reached") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
                
    return None

In [432]:
def depth_first_search_2(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    explored = []

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        explored.append(node.state)

        for action in actions(node.state):
            child = child_node(node, action)

            in_explored = False
            for s in explored:
                if s == child.state:
                    in_explored = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_explored and not in_frontier:
                if goal_test(child.state):
                    return solution(child)

                frontier.append(child)
                
        gui.log_step(node, frontier, explored, "Explored") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
                
    return None

In [433]:
def depth_limited_search_1(initial_state, limit, gui):
    node = Node(initial_state)
    
    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = []
    cutoff_occurred = False

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        # Nếu đạt đến giới hạn độ sâu, không mở rộng (expand) node này nữa
        if node.cost >= limit:
            cutoff_occurred = True
            gui.log_step(node, frontier, reached, "Reached")
            continue

        for action in actions(node.state):
            child = child_node(node, action)
            
            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        gui.log_step(node, frontier, reached, "Reached")
                
    return "CUTOFF" if cutoff_occurred else None

In [434]:
def depth_limited_search_2(initial_state, limit, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    explored = []
    cutoff_occurred = False

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        explored.append(node.state)

        # Nếu đạt đến giới hạn độ sâu, không mở rộng (expand) node này nữa
        if node.cost >= limit:
            cutoff_occurred = True
            gui.log_step(node, frontier, explored, "Explored")
            continue

        for action in actions(node.state):
            child = child_node(node, action)

            in_explored = False
            for s in explored:
                if s == child.state:
                    in_explored = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_explored and not in_frontier:
                if goal_test(child.state):
                    return solution(child)

                frontier.append(child)
                
        gui.log_step(node, frontier, explored, "Explored")
                
    return "CUTOFF" if cutoff_occurred else None

In [435]:
def iterative_deepening_search_1(initial_state, gui):
    max_depth = 50 # Giới hạn độ sâu an toàn để tránh lặp vô hạn
    
    for depth in range(max_depth):
        # Thông báo ra log GUI cho dễ theo dõi
        gui.log_text.insert(tk.END, f"\n{'*'*14} Depth = {depth} {'*'*14}\n")
        
        # Reset lại Node counter và log hiển thị để bắt đầu vòng lặp mới sạch sẽ
        Node.reset_counter()
        if hasattr(gui, 'explored_names_log'):
            gui.explored_names_log = []

        result = depth_limited_search_1(initial_state, depth, gui)
        
        # Nếu tìm thấy kết quả hợp lệ (không phải bị cutoff và không phải thất bại)
        if result != "CUTOFF" and result is not None:
            return result
        
        # Nếu duyệt hết toàn bộ không gian trạng thái mà không bị cutoff -> Thất bại
        if result is None:
            return None
            
    return None

In [436]:
def iterative_deepening_search_2(initial_state, gui):
    max_depth = 50 
    
    for depth in range(max_depth):
        gui.log_text.insert(tk.END, f"\n{'*'*14} Depth = {depth} {'*'*14}\n")
        
        Node.reset_counter()
        if hasattr(gui, 'explored_names_log'):
            gui.explored_names_log = []

        result = depth_limited_search_2(initial_state, depth, gui)
        
        if result != "CUTOFF" and result is not None:
            return result
            
        if result is None:
            return None
            
    return None

In [437]:
def uniform_cost_search(initial_state, gui):
    node = Node(initial_state)
    
    # Ghi đè cost của root bằng số lượng bụi hiện tại
    node.cost = count_dust(node.state) 
    
    if goal_test(node.state):
        return solution(node)

    # Priority Queue: Lưu tuple (cost, name, node) để heap tự động sắp xếp theo cost
    # Thuộc tính name được đưa vào để tie-break (tránh lỗi khi 2 node có cùng cost)
    frontier = []
    heapq.heappush(frontier, (node.cost, node.name, node))
    reached = []

    while len(frontier) > 0:
        # Lấy node có cost nhỏ nhất ra khỏi Priority Queue
        current_cost, _, node = heapq.heappop(frontier)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            # Tính Path Cost mới: Cost cha + Số lượng bụi của Node con (số ô sai so với GOAL)
            child.cost = node.cost + count_dust(child.state)

            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for i, (f_cost, f_name, f_node) in enumerate(frontier):
                if f_node.state == child.state:
                    in_frontier = True
                    # Nếu trạng thái đã có trong Frontier nhưng chi phí mới tốt hơn -> Thay thế node cũ
                    if child.cost < f_cost:
                        frontier[i] = (child.cost, child.name, child)
                        heapq.heapify(frontier) # Cấu trúc lại cấu trúc heap sau khi thay đổi
                    break

            if not in_reached and not in_frontier:
                heapq.heappush(frontier, (child.cost, child.name, child))
                
        # Format lại frontier (chỉ lấy mảng node) để đưa vào hàm log_step in ra giao diện
        frontier_nodes = [item[2] for item in frontier]
        gui.log_step(node, frontier_nodes, reached, "Reached")
        
    return None

In [438]:
def greedy_search(initial_state, gui):
    node = Node(initial_state)
    
    # h(n) = số lượng bụi hiện tại (số ô sai so với trạng thái đích)
    node.cost = count_dust(node.state) 
    
    if goal_test(node.state):
        return solution(node)

    # Priority Queue: Lưu tuple (h(n), name, node) để heap tự động sắp xếp theo h(n)
    frontier = []
    heapq.heappush(frontier, (node.cost, node.name, node))
    reached = []

    while len(frontier) > 0:
        # Lấy node có cost h(n) nhỏ nhất ra khỏi Priority Queue
        current_cost, _, node = heapq.heappop(frontier)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            # Cost chỉ là h(n) của node hiện tại đang xét, KHÔNG cộng dồn cost của node cha
            child.cost = count_dust(child.state)

            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for f_cost, f_name, f_node in frontier:
                if f_node.state == child.state:
                    in_frontier = True
                    break

            if not in_reached and not in_frontier:
                heapq.heappush(frontier, (child.cost, child.name, child))
                
        # Format lại frontier để đưa vào hàm log_step in ra giao diện
        frontier_nodes = [item[2] for item in frontier]
        gui.log_step(node, frontier_nodes, reached, "Reached")
        
    return None

In [439]:
def get_dust_positions(matrix):
    dusts = []
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            if matrix[i][j] == 1:
                dusts.append((i, j))
    return dusts

def heuristic_cost(state):
    robot_pos = find_robot(state)
    dusts = get_dust_positions(state)
    
    if not dusts or not robot_pos:
        return 0
        
    rx, ry = robot_pos
    min_distance = float('inf')
    
    # Tìm khoảng cách Manhattan đến hạt bụi gần nhất
    for dx, dy in dusts:
        distance = abs(rx - dx) + abs(ry - dy)
        if distance < min_distance:
            min_distance = distance
            
    # h(n): Manhattan gần nhất + (tổng số bụi - 1)
    return min_distance + (len(dusts) - 1)

In [440]:
def a_star_search(initial_state, gui):
    node = Node(initial_state)
    
    # g(n): Số ô sai (số lượng bụi), có kế thừa (tích lũy)
    node.g = count_dust(node.state)
    # h(n): Manhattan gần nhất + (tổng số bụi - 1), không kế thừa
    node.h = heuristic_cost(node.state)
    # f(n) = g(n) + h(n). Gán vào node.cost để dùng chung hàm log_step
    node.cost = node.g + node.h 
    
    if goal_test(node.state):
        return solution(node)

    # Priority Queue: Lưu tuple (f(n), name, node) để tự động sắp xếp theo f(n)
    frontier = []
    heapq.heappush(frontier, (node.cost, node.name, node))
    reached = []

    while len(frontier) > 0:
        # Lấy node có f(n) nhỏ nhất ra khỏi Priority Queue
        current_cost, _, node = heapq.heappop(frontier)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            child.g = node.g + count_dust(child.state)
            
            child.h = heuristic_cost(child.state)
            
            # Tính f(n)
            child.cost = child.g + child.h

            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True
                    break

            in_frontier = False
            for i, (f_cost, f_name, f_node) in enumerate(frontier):
                if f_node.state == child.state:
                    in_frontier = True
                    # Nếu trạng thái đã có trong Frontier nhưng chi phí mới tốt hơn -> Thay thế node cũ
                    if child.cost < f_cost:
                        frontier[i] = (child.cost, child.name, child)
                        heapq.heapify(frontier) # Cấu trúc lại cấu trúc heap sau khi thay đổi
                    break

            if not in_reached and not in_frontier:
                heapq.heappush(frontier, (child.cost, child.name, child))
                
        # Format lại frontier để đưa vào hàm log_step in ra giao diện
        frontier_nodes = [item[2] for item in frontier]
        gui.log_step(node, frontier_nodes, reached, "Reached")
        
    return None

In [441]:
def bounded_a_star_search(initial_state, threshold_I, gui):
    node = Node(initial_state)
    
    # Khởi tạo chi phí giống hệt A*
    node.g = count_dust(node.state)
    node.h = heuristic_cost(node.state)
    node.cost = node.g + node.h 
    
    if goal_test(node.state):
        return solution(node), None

    frontier = []
    heapq.heappush(frontier, (node.cost, node.name, node))
    reached = []
    
    # next_I lưu lại mức chi phí (f(n)) nhỏ nhất vượt qua ngưỡng I hiện tại
    next_I = float('inf') 
    cutoff_occurred = False

    while len(frontier) > 0:
        current_cost, _, node = heapq.heappop(frontier)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node), None

        for action in actions(node.state):
            child = child_node(node, action)
            
            # Tính toán chi phí cho node con theo logic của A*
            child.g = node.g + count_dust(child.state)
            child.h = heuristic_cost(child.state)
            child.cost = child.g + child.h

            # Ràng buộc không đưa vào frontier nếu lớn hơn chi phí I
            if child.cost > threshold_I:
                cutoff_occurred = True
        
                if child.cost < next_I:
                    next_I = child.cost
                continue 

            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True
                    break

            in_frontier = False
            for i, (f_cost, f_name, f_node) in enumerate(frontier):
                if f_node.state == child.state:
                    in_frontier = True
                    if child.cost < f_cost:
                        frontier[i] = (child.cost, child.name, child)
                        heapq.heapify(frontier)
                    break

            if not in_reached and not in_frontier:
                heapq.heappush(frontier, (child.cost, child.name, child))
                
        # Format lại frontier để đưa vào hàm log_step in ra giao diện
        frontier_nodes = [item[2] for item in frontier]
        gui.log_step(node, frontier_nodes, reached, "Reached")
        
    return ("CUTOFF", next_I) if cutoff_occurred else (None, None)

def ida_star_search(initial_state, gui):
    # Khởi tạo chi phí I ban đầu bằng số ô sai (số hạt bụi) của trạng thái khởi đầu
    threshold_I = count_dust(initial_state)
    
    max_iterations = 50 # Giới hạn chống treo máy
    
    for i in range(max_iterations):
        gui.log_text.insert(tk.END, f"\n{'*'*14} Chi phí I = {threshold_I} {'*'*14}\n")
        
        Node.reset_counter()
        if hasattr(gui, 'explored_names_log'):
            gui.explored_names_log = []

        # Chạy A* có giới hạn ngưỡng I
        result, next_I = bounded_a_star_search(initial_state, threshold_I, gui)
        
        if result != "CUTOFF" and result is not None:
            return result
            
        if result is None:
            return None
            
        # Cập nhật chi phí I cho vòng lặp tiếp theo bằng min f(n) của các node bị loại
        if next_I == float('inf'):
            break
        threshold_I = next_I
        
    return None

In [442]:
def simple_hill_climbing(initial_state, gui):
    current_node = Node(initial_state)
    current_node.cost = count_dust(current_node.state)
    
    gui.log_tree_node(current_node, status="(Khởi tạo)")
    
    while True:
        if goal_test(current_node.state):
            return solution(current_node)
        
        better_neighbor_found = False
        
        for action in actions(current_node.state):
            child = child_node(current_node, action)
            child.cost = count_dust(child.state)
            
            if child.cost < current_node.cost:
                gui.log_tree_node(child, status="---> CHỌN")
                current_node = child
                better_neighbor_found = True
                break # Thoát vòng lặp lân cận, đi tới node này ngay lập tức
            else:
                gui.log_tree_node(child, status="- Bỏ qua")
                
        if not better_neighbor_found:
            gui.log_local_minimum(current_node)
            return "LOCAL_MINIMUM"

In [443]:
def steepest_ascent_hill_climbing(initial_state, gui):
    current_node = Node(initial_state)
    current_node.cost = count_dust(current_node.state)
    
    gui.log_tree_node(current_node, status="(Khởi tạo)")
    
    while True:
        if goal_test(current_node.state):
            return solution(current_node)
        
        best_neighbor = None
        lowest_cost = float('inf')
        neighbors = [] 
        
        # 1. Sinh và đánh giá tất cả lân cận
        for action in actions(current_node.state):
            child = child_node(current_node, action)
            child.cost = count_dust(child.state)
            neighbors.append(child)
            
            if child.cost < lowest_cost:
                lowest_cost = child.cost
                best_neighbor = child
                
        # 2. Lựa chọn node đi tiếp và in Log
        if best_neighbor and lowest_cost < current_node.cost:
            for n in neighbors:
                status = "---> CHỌN" if n == best_neighbor else "- Bỏ qua"
                gui.log_tree_node(n, status)
            current_node = best_neighbor
        else:
            for n in neighbors:
                gui.log_tree_node(n, status="- Bỏ qua")
                
            gui.log_local_minimum(current_node)
            return "LOCAL_MINIMUM"

In [444]:
def stochastic_hill_climbing(initial_state, gui):
    current_node = Node(initial_state)
    current_node.cost = count_dust(current_node.state)
    
    gui.log_tree_node(current_node, status="(Khởi tạo)")
    
    while True:
        if goal_test(current_node.state):
            return solution(current_node)
        
        better_neighbors = []
        all_neighbors = []
        
        # 1. Sinh và phân loại lân cận
        for action in actions(current_node.state):
            child = child_node(current_node, action)
            child.cost = count_dust(child.state)
            all_neighbors.append(child)
            
            if child.cost < current_node.cost:
                better_neighbors.append(child)
                
        # 2. Quyết định hướng đi: Chọn ngẫu nhiên từ danh sách tốt hơn
        if better_neighbors:
            chosen_node = random.choice(better_neighbors)
            
            for n in all_neighbors:
                if n == chosen_node:
                    gui.log_tree_node(n, status="---> CHỌN (Ngẫu nhiên)")
                elif n in better_neighbors:
                    gui.log_tree_node(n, status="- Tốt hơn (Không chọn)")
                else:
                    gui.log_tree_node(n, status="- Bỏ qua")
                    
            current_node = chosen_node
        else:
            for n in all_neighbors:
                gui.log_tree_node(n, status="- Bỏ qua")
                
            gui.log_local_minimum(current_node)
            return "LOCAL_MINIMUM"

In [445]:
def random_restart_stochastic_hill_climbing(initial_state, gui):
    max_restart = 20
    
    for i in range(1, max_restart + 1):
        gui.log_message(f"\n{'*'*15} Lần chạy lại (Restart) thứ {i} {'*'*15}")
        Node.reset_counter() # Reset lại tên node cho dễ nhìn
        
        current_node = Node(initial_state)
        current_node.cost = count_dust(current_node.state)
        
        gui.log_tree_node(current_node, status="(Khởi tạo)")
        
        stuck = False
        while not stuck:
            if goal_test(current_node.state):
                gui.log_message(f"\n[!] THÀNH CÔNG: Đã tìm thấy Goal tại lượt lặp thứ {i}.")
                return solution(current_node)
            
            better_neighbors = []
            all_neighbors = []
            
            # Sinh và đánh giá lân cận
            for action in actions(current_node.state):
                child = child_node(current_node, action)
                child.cost = count_dust(child.state)
                all_neighbors.append(child)
                
                if child.cost < current_node.cost:
                    better_neighbors.append(child)
                    
            # Quyết định hướng đi
            if better_neighbors:
                chosen_node = random.choice(better_neighbors)
                
                for n in all_neighbors:
                    if n == chosen_node:
                        gui.log_tree_node(n, status="---> CHỌN (Ngẫu nhiên)")
                    elif n in better_neighbors:
                        gui.log_tree_node(n, status="- Tốt hơn")
                    else:
                        gui.log_tree_node(n, status="- Bỏ qua")
                        
                current_node = chosen_node
            else:
                for n in all_neighbors:
                    gui.log_tree_node(n, status="- Bỏ qua")
                    
                gui.log_message(f"\n[!] Lượt {i} bị kẹt tại node {current_node.name}. Tiến hành restart...")
                stuck = True # Đánh dấu kẹt để thoát vòng lặp while, nhảy sang lượt for tiếp theo
                
    # Nếu chạy hết vòng lặp for mà vẫn không return
    gui.log_message(f"\n[!] THẤT BẠI: Đã chạy hết {max_restart} lần lặp nhưng vẫn không tìm được đích.")
    return "LOCAL_MINIMUM"

In [446]:
def local_beam_search(initial_state, gui, k=2):
    start_node = Node(initial_state)
    start_node.cost = count_dust(start_node.state)
    
    gui.log_message(f"Khởi tạo chùm với k={k}:")
    gui.log_tree_node(start_node, status="(Gốc)")
    
    if goal_test(start_node.state):
        gui.log_tree_node(start_node, status="---> TÌM THẤY ĐÍCH")
        return solution(start_node)
        
    current_nodes = [start_node]
    max_iter = 100 # Giới hạn vòng lặp chống treo máy
    
    for i in range(1, max_iter + 1):
        neighbor_nodes = []
        
        # 1. Sinh TẤT CẢ trạng thái lân cận của các node trong chùm
        for node in current_nodes:
            for action in actions(node.state):
                child = child_node(node, action)
                child.cost = count_dust(child.state)
                neighbor_nodes.append(child)
                
        # 2. Nếu không thể sinh thêm lân cận (Bị kẹt)
        if not neighbor_nodes:
            gui.log_message("\n[!] Bị kẹt: Không thể sinh thêm lân cận.")
            return "LOCAL_MINIMUM"
            
        # 3. Kiểm tra đích trong tập lân cận vừa sinh
        for neighbor in neighbor_nodes:
            if goal_test(neighbor.state):
                gui.log_message(f"\n--- Bước {i} ---")
                gui.log_tree_node(neighbor, status="---> TÌM THẤY ĐÍCH")
                return solution(neighbor)
                
        # 4. Lựa chọn chùm: Sắp xếp theo h(n) tốt dần và cắt lấy k phần tử
        neighbor_nodes.sort(key=lambda x: (x.cost, x.name))
        current_nodes = neighbor_nodes[:k] 
        
        # Gọi hàm GUI in ra k phần tử vừa được giữ lại
        gui.log_beam_step(i, current_nodes)
        
    gui.log_message(f"\n[!] Dừng thuật toán: Đạt giới hạn {max_iter} vòng lặp.")
    return "LOCAL_MINIMUM"

In [447]:
def simulated_annealing(initial_state, gui, T0=100, Tmin=1, alpha=0.95):
    # Khởi tạo: current state = start
    current_node = Node(initial_state)
    current_node.cost = heuristic_cost(current_node.state) # h(n) = Manhattan gần nhất + (tổng số bụi - 1)
    
    gui.log_message(f"\n{'*'*15} Bắt đầu Simulated Annealing {'*'*15}")
    gui.log_message(f"Tham số: T0={T0}, Tmin={Tmin}, alpha={alpha}\n")
    gui.log_tree_node(current_node, status="(Khởi tạo)")
    
    # T = T0
    T = T0
    
    # while T > Tmin:
    while T > Tmin:
        # if current state == goal: return current state
        if goal_test(current_node.state):
            gui.log_message(f"\n[!] THÀNH CÔNG: Đã tìm thấy Goal tại nhiệt độ T = {T:.4f}")
            return solution(current_node)
        
        # Lấy tất cả các trạng thái lân cận
        possible_actions = actions(current_node.state)
        if not possible_actions:
            break # Kẹt cứng không có đường đi (rất hiếm xảy ra)
            
        # next state = RandomNeighbor(current state)
        action = random.choice(possible_actions)
        next_node = child_node(current_node, action)
        next_node.cost = heuristic_cost(next_node.state)
        
        # Δ = h(next state) - h(current state)
        delta = next_node.cost - current_node.cost
        
        # if Δ < 0: current state = next state
        if delta < 0:
            current_node = next_node
            gui.log_tree_node(next_node, status=f"---> CHỌN (Tốt hơn, Δ={delta}, T={T:.2f})")
        else:
            # else: p = exp(-Δ / T)
            p = math.exp(-delta / T)
            # if Random(0,1) < p: current state = next state
            if random.random() < p:
                current_node = next_node
                gui.log_tree_node(next_node, status=f"---> CHỌN (p={p:.4f}, Δ={delta}, T={T:.2f})")
            else:
                gui.log_tree_node(next_node, status=f"- Bỏ qua (p={p:.4f}, Δ={delta}, T={T:.2f})")
        
        # T = α * T
        T = alpha * T
        
    # Vòng lặp kết thúc mà chưa đạt Goal (Hạ nhiệt độ xuống dưới Tmin)
    gui.log_message(f"\n[!] Thuật toán dừng vì nhiệt độ đã hạ xuống ngưỡng Tmin ({T:.4f} <= {Tmin})")
    gui.log_local_minimum(current_node)
    return "LOCAL_MINIMUM"

In [448]:
def sensorless_bfs(initial_states, gui):
    node = BeliefNode(initial_states)
    
    if bs_goal_test(node.states):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = [] # Lưu chuỗi string của BS

    # Không gian hành động mù (luôn thử 4 hướng)
    possible_actions = ["UP", "DOWN", "LEFT", "RIGHT"]

    while len(frontier) > 0:
        node = frontier.popleft()
        reached.append(bs_to_str(node.states))
        
        if bs_goal_test(node.states):
            return solution(node)

        for action in possible_actions:
            child = bs_child_node(node, action)
            child_str = bs_to_str(child.states)
            
            in_reached = child_str in reached
            in_frontier = any(bs_to_str(n.states) == child_str for n in frontier)

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        gui.log_bs_step(node, frontier, reached, "Reached")
        
    return None

In [449]:
def partial_obs_bfs(initial_states, gui):
    node = BeliefNode(initial_states)
    
    if bs_goal_test(node.states):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = [] # Lưu chuỗi string của BS để đánh dấu đã duyệt

    # Không gian hành động di chuyển
    possible_actions = ["UP", "DOWN", "LEFT", "RIGHT"]

    while len(frontier) > 0:
        node = frontier.popleft()
        reached.append(bs_to_str(node.states))
        
        if bs_goal_test(node.states):
            return solution(node)

        for action in possible_actions:
            # Sinh trạng thái con 
            child = bs_child_node(node, action)
            child_str = bs_to_str(child.states)
            
            # Kiểm tra trùng lặp
            in_reached = child_str in reached
            in_frontier = any(bs_to_str(n.states) == child_str for n in frontier)

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        # Gọi hàm log chuyên dụng của Belief State
        gui.log_bs_step(node, frontier, reached, "Reached")
        
    return None

In [450]:
class CleaningRobotGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Cleaning Robot")
        
        try:
            self.root.state('zoomed')  # Hoạt động tốt nhất trên Windows
        except Exception:
            try:
                self.root.attributes('-zoomed', True)  # Hoạt động trên Linux (X11)
            except Exception:
                # Fallback cho macOS hoặc các hệ điều hành khác: Lấy kích thước thực của màn hình
                w = self.root.winfo_screenwidth()
                h = self.root.winfo_screenheight()
                self.root.geometry(f"{w}x{h}+0+0")
                
        self.root.configure(bg="#F5F5F5")

        self.style = ttk.Style()
        self.style.theme_use('clam')
        self.style.configure('TButton', font=('Segoe UI', 10), padding=5)
        self.style.configure('TLabel', font=('Segoe UI', 10), background="#F5F5F5")

        self.initial_matrix = []
        self.current_matrix = []
        self.is_running = False
        self.canvas_dim = 360 # Cố định kích thước khung vẽ ma trận
        
        self.size_var = tk.StringVar(value="3x3") # Lưu kích thước ma trận (mặc định là 3x3)

        self.setup_ui()
        self.generate_map()

    def setup_ui(self):
        # 1. KHU VỰC PHÍA DƯỚI (Solution Box)
        self.bottom_frame = tk.Frame(self.root, bg="#F5F5F5")
        self.bottom_frame.pack(side=tk.BOTTOM, fill=tk.X, padx=10, pady=(0, 10))
        
        ttk.Label(self.bottom_frame, text="Solution Actions:", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 5))
        self.sol_text = tk.Text(self.bottom_frame, height=3, font=("Consolas", 10), bd=1, relief="solid", bg="#FAFAFA")
        self.sol_text.pack(fill=tk.X)
        self.sol_text.config(state=tk.DISABLED)

        # 2. KHU VỰC PHÍA TRÊN (Chứa Controls, Canvas, Log)
        self.top_frame = tk.Frame(self.root, bg="#F5F5F5")
        self.top_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=10, pady=10)

        # Cột 1: Controls (Bảng điều khiển nút bấm & Tùy chọn)
        self.control_frame = tk.Frame(self.top_frame, bg="#F5F5F5")
        self.control_frame.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 20))

        # PHẦN TÙY CHỌN BẢN ĐỒ
        ttk.Label(self.control_frame, text="Map Settings", font=('Segoe UI', 11, 'bold')).pack(anchor="w", pady=(0, 5))
        map_setting_frame = tk.Frame(self.control_frame, bg="#F5F5F5")
        map_setting_frame.pack(fill=tk.X, pady=(0, 5))
        
        self.size_combo = ttk.Combobox(map_setting_frame, textvariable=self.size_var, values=["3x3", "4x4", "5x5"], state="readonly", width=6)
        self.size_combo.pack(side=tk.LEFT, padx=(0, 5))
        self.size_combo.bind("<<ComboboxSelected>>", lambda event: self.generate_map())
        
        ttk.Button(map_setting_frame, text="Generate Map", command=self.generate_map, width=12).pack(side=tk.LEFT)

        ttk.Separator(self.control_frame, orient='horizontal').pack(fill='x', pady=8)

        # CHIA KHU VỰC THUẬT TOÁN THÀNH 2 CỘT
        button_columns_frame = tk.Frame(self.control_frame, bg="#F5F5F5")
        button_columns_frame.pack(fill=tk.BOTH, expand=True)

        col1 = tk.Frame(button_columns_frame, bg="#F5F5F5")
        col1.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 10), anchor="n")

        col2 = tk.Frame(button_columns_frame, bg="#F5F5F5")
        col2.pack(side=tk.LEFT, fill=tk.Y, padx=(10, 0), anchor="n")

        # NÚT BẤM CỘT 1
        ttk.Label(col1, text="Uninformed Search", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 5))
        ttk.Button(col1, text="BFS 1", command=lambda: self.run_algo("BFS1"), width=16).pack(pady=1)
        ttk.Button(col1, text="BFS 2", command=lambda: self.run_algo("BFS2"), width=16).pack(pady=1)
        ttk.Button(col1, text="DFS 1", command=lambda: self.run_algo("DFS1"), width=16).pack(pady=1)
        ttk.Button(col1, text="DFS 2", command=lambda: self.run_algo("DFS2"), width=16).pack(pady=1)
        ttk.Button(col1, text="IDS 1", command=lambda: self.run_algo("IDS1"), width=16).pack(pady=1)
        ttk.Button(col1, text="IDS 2", command=lambda: self.run_algo("IDS2"), width=16).pack(pady=1)
        ttk.Button(col1, text="UCS", command=lambda: self.run_algo("UCS"), width=16).pack(pady=1)
        
        ttk.Separator(col1, orient='horizontal').pack(fill='x', pady=5)
        ttk.Label(col1, text="Informed Search", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 2))
        ttk.Button(col1, text="Greedy Search", command=lambda: self.run_algo("GS"), width=16).pack(pady=1)
        ttk.Button(col1, text="A* Search", command=lambda: self.run_algo("A*"), width=16).pack(pady=1)
        ttk.Button(col1, text="IDA* Search", command=lambda: self.run_algo("IDA*"), width=16).pack(pady=1)

        # NÚT BẤM CỘT 2
        ttk.Label(col2, text="Local Search", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 5))
        ttk.Button(col2, text="Simple HC", command=lambda: self.run_algo("SHC"), width=16).pack(pady=1)
        ttk.Button(col2, text="Steepest HC", command=lambda: self.run_algo("SAHC"), width=16).pack(pady=1)
        ttk.Button(col2, text="Stoc HC", command=lambda: self.run_algo("StocHC"), width=16).pack(pady=1)
        ttk.Button(col2, text="Random Restart", command=lambda: self.run_algo("RRHC"), width=16).pack(pady=1)
        ttk.Button(col2, text="Local Beam Search", command=lambda: self.run_algo("LBS"), width=16).pack(pady=1)
        ttk.Button(col2, text="Simulated Annealing", command=lambda: self.run_algo("SA"), width=16).pack(pady=1)

        ttk.Separator(col2, orient='horizontal').pack(fill='x', pady=5)
        ttk.Label(col2, text="Sensorless Env", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 2))
        ttk.Button(col2, text="Gen 2 States (BS)", command=self.generate_belief_states, width=16).pack(pady=1)
        ttk.Button(col2, text="Sensorless Search", command=self.run_sensorless_algo, width=16).pack(pady=1)

        ttk.Separator(col2, orient='horizontal').pack(fill='x', pady=5)
        ttk.Label(col2, text="Partial Obs Env", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 2))
        ttk.Button(col2, text="Gen Partial BS", command=self.generate_partial_belief_states, width=16).pack(pady=1)
        ttk.Button(col2, text="Partial Obs Search", command=self.run_partial_obs_algo, width=16).pack(pady=1)

        # Cột 2: Canvas (Khung vẽ bản đồ ma trận)
        self.canvas_frame = tk.Frame(self.top_frame, bg="white", bd=1, relief="solid")
        self.canvas_frame.pack(side=tk.LEFT, padx=10)
        self.canvas = tk.Canvas(self.canvas_frame, width=self.canvas_dim, height=self.canvas_dim, bg="white", highlightthickness=0)
        self.canvas.pack(padx=2, pady=2)

        # Cột 3: Log (Khung hiển thị lịch trình tìm kiếm)
        self.log_frame = tk.Frame(self.top_frame, bg="#F5F5F5")
        self.log_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(20, 0))
        
        ttk.Label(self.log_frame, text="Search Execution Log", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 5))
        self.log_text = scrolledtext.ScrolledText(self.log_frame, width=50, font=("Consolas", 9), bd=1, relief="solid")
        self.log_text.pack(fill=tk.BOTH, expand=True)

    def generate_map(self):
        if self.is_running: return
        
        size_str = self.size_var.get()
        size = int(size_str.split('x')[0]) # Tách lấy số đầu tiên (vd: "4x4" -> 4)
        
        obstacle_mapping = {3: 2, 4: 3, 5: 6}
        num_obstacles = obstacle_mapping.get(size, 2)
        
        # Giới hạn số lượng hạt bụi ngẫu nhiên sao cho phù hợp với từng kích thước grid
        max_dust_mapping = {3: 4, 4: 6, 5: 8}
        num_dust = random.randint(1, max_dust_mapping.get(size, 4))
        
        # 1. Khởi tạo toàn bộ ma trận là 0 (Sạch)
        self.initial_matrix = [[0 for _ in range(size)] for _ in range(size)]
        
        # 2. Đặt robot ngẫu nhiên (Giá trị 2)
        self.initial_matrix[random.randint(0, size-1)][random.randint(0, size-1)] = 2
        
        # 3. Đặt chướng ngại vật ngẫu nhiên (Giá trị 3)
        obs_placed = 0
        while obs_placed < num_obstacles:
            rx, ry = random.randint(0, size-1), random.randint(0, size-1)
            if self.initial_matrix[rx][ry] == 0:
                self.initial_matrix[rx][ry] = 3
                obs_placed += 1
                
        # 4. Đặt bụi ngẫu nhiên (Giá trị 1)
        dust_placed = 0
        while dust_placed < num_dust:
            rx, ry = random.randint(0, size-1), random.randint(0, size-1)
            if self.initial_matrix[rx][ry] == 0:
                self.initial_matrix[rx][ry] = 1
                dust_placed += 1
                
        self.current_matrix = [row[:] for row in self.initial_matrix]
        self.draw_grid(self.current_matrix)
        
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, f"Generated map with {num_dust} dust particles.\nReady for search.\n")
        self.update_solution_text("")

    def draw_grid(self, matrix):
        self.canvas.delete("all")
        size = len(matrix)
        cell_size = self.canvas_dim / size
        
        colors = {0: "#A5D6A7", 1: "#E0E0E0", 2: "#64B5F6", 3: "#E57373"}
        labels = {0: "Clean", 1: "Dust", 2: "Robot", 3: "Wall"}
        
        for i in range(size):
            for j in range(size):
                val = matrix[i][j]
                x1, y1 = j * cell_size, i * cell_size
                x2, y2 = x1 + cell_size, y1 + cell_size
                
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=colors[val], outline="#FFFFFF", width=2)
                
                # Cỡ chữ tự động co giãn linh hoạt theo kích thước ma trận để không bị tràn ô
                font_size = 12 if size <= 3 else max(7, int(12 - (size - 3) * 1.8))
                self.canvas.create_text(x1 + cell_size/2, y1 + cell_size/2, 
                                        text=labels[val], font=("Segoe UI", int(font_size), "bold"), fill="#424242" if val in [0, 1] else "#FFFFFF")
                
        self.root.update_idletasks()
        
    def state_to_str(self, state):
        res = []
        for row in state:
            # Giữ nguyên giá trị gốc của ma trận (chỉ ép kiểu về string để nối chuỗi)
            res.append("[" + ",".join(str(val) for val in row) + "]")
        return "[" + ",".join(res) + "]"
    
    def short_action(self, action):
        if not action: return "-"
        return {"UP": "U", "DOWN": "D", "LEFT": "L", "RIGHT": "R"}.get(action, action)

    def log_step(self, current_node, frontier, closed_list, closed_name):
        # Lưu trữ danh sách tên [A.State, B.State]
        if not hasattr(self, 'explored_names_log'):
            self.explored_names_log = []
            
        explored_val = current_node.name + ".State"
        if explored_val not in self.explored_names_log:
            self.explored_names_log.append(explored_val)

        # Định dạng Frontier: {[State], Parent, Action, Cost} NodeName
        frontier_strs = []
        for n in frontier:
            state_str = self.state_to_str(n.state)
            parent_name = n.parent.name if n.parent else "-"
            act = self.short_action(n.action)
            cost = n.cost
            frontier_strs.append(f"  {{{state_str}, {parent_name}, {act}, {cost}}} {n.name}")
        
        frontier_display = "[\n" + ",\n".join(frontier_strs) + "\n]" if frontier_strs else "[]"
        explored_display = "[" + ", ".join(self.explored_names_log) + "]"

        # Xuất ra Log
        log_msg =  f"Node     : {current_node.name}\n"
        log_msg += f"Frontier : {frontier_display}\n"
        log_msg += f"Explored : {explored_display}\n"
        log_msg += "-" * 40 + "\n"
        
        self.log_text.insert(tk.END, log_msg)
        self.log_text.see(tk.END)
        self.root.update()
        
    def log_tree_node(self, node, status=""):
        """In ra một nhánh của cây tìm kiếm local"""
        depth = len(solution(node)) if node.parent else 0
        indent = "    " * depth
        action_str = f"[Action: {node.action}] " if node.action else ""
        prefix = f"{indent}|-- " if depth > 0 else ""
        
        self.log_text.insert(tk.END, f"{prefix}{node.name} (Cost={node.cost}) {action_str}{status}\n")
        self.root.update()

    def log_local_minimum(self, node):
        """In ra bảng tổng kết khi thuật toán bị kẹt ở Local Minimum"""
        best_state_str = self.state_to_str(node.state)
        path = solution(node)
        path_str = " -> ".join(path) if path else "Chưa di chuyển (Kẹt ngay tại gốc)"
        
        log_msg = (
            f"\n[!] Dừng thuật toán - Kẹt ở Local Minimum\n"
            f"{'='*40}\n"
            f"[*] THÔNG TIN NODE TỐI ƯU NHẤT TÌM ĐƯỢC:\n"
            f"    - Tên Node : {node.name}\n"
            f"    - Chi phí  : h(n) = {node.cost} (Số hạt bụi còn lại)\n"
            f"    - Actions  : {path_str}\n"
            f"    - Trạng thái:\n      {best_state_str}\n"
            f"{'='*40}\n"
        )
        self.log_text.insert(tk.END, log_msg)
        self.root.update()
        
    def log_message(self, msg):
        """Hàm đa năng để in các thông báo sự kiện (Restart, Kẹt, Thành công...)"""
        self.log_text.insert(tk.END, f"{msg}\n")
        self.root.update()

    def log_beam_step(self, step, nodes):
        """In ra danh sách các node được giữ lại trong một bước của Beam Search"""
        self.log_text.insert(tk.END, f"--- Bước {step} (Giữ lại {len(nodes)} node tốt nhất) ---\n")
        for c_node in nodes:
            action_str = f"[Action: {c_node.action}] " if c_node.action else ""
            parent_str = c_node.parent.name if c_node.parent else "None"
            self.log_text.insert(tk.END, f" |-- {c_node.name} (Từ {parent_str}) {action_str}h={c_node.cost}\n")
        self.root.update()

    def update_solution_text(self, text):
        self.sol_text.config(state=tk.NORMAL)
        self.sol_text.delete(1.0, tk.END)
        self.sol_text.insert(tk.END, text)
        self.sol_text.config(state=tk.DISABLED)

    def run_algo(self, algo_name):
        if self.is_running: return
        self.is_running = True
        
        Node.reset_counter()
        self.explored_names_log = []
        
        self.current_matrix = [row[:] for row in self.initial_matrix]
        self.draw_grid(self.current_matrix)
        
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, f"[{algo_name}] Execution Started\n{'='*40}\n")
        
        # In ra node khởi tạo (Initial Node) trước khi bắt đầu thuật toán
        init_state_str = self.state_to_str(self.current_matrix)
        
        # Xác định cost khởi tạo
        if algo_name in ["UCS", "GS", "SHC", "SAHC", "StocHC", "RRHC", "LBS", "SA"]: 
            init_cost = count_dust(self.current_matrix)
        elif algo_name in ["A*", "IDA*"]: 
            init_g = count_dust(self.current_matrix)
            init_h = heuristic_cost(self.current_matrix)
            init_cost = init_g + init_h
        else:
            init_cost = 0
            
        if algo_name in ["SHC", "SAHC", "StocHC", "RRHC", "LBS", "SA"]:
            self.log_text.insert(tk.END, f"Tree Diagram (Heuristic: h(n) = count_dust):\n")
        else:
            self.log_text.insert(tk.END, f"Initial Node:\n  {{{init_state_str}, _, _, {init_cost}}} A\n")
            self.log_text.insert(tk.END, f"{'-'*40}\n")
        
        self.update_solution_text("Searching...")
        
        path = None
        if algo_name == "BFS1": path = breadth_first_search_1(self.current_matrix, self)
        elif algo_name == "BFS2": path = breadth_first_search_2(self.current_matrix, self)
        elif algo_name == "DFS1": path = depth_first_search_1(self.current_matrix, self)
        elif algo_name == "DFS2": path = depth_first_search_2(self.current_matrix, self)
        elif algo_name == "IDS1": path = iterative_deepening_search_1(self.current_matrix, self)
        elif algo_name == "IDS2": path = iterative_deepening_search_2(self.current_matrix, self)
        elif algo_name == "UCS": path = uniform_cost_search(self.current_matrix, self)
        elif algo_name == "GS": path = greedy_search(self.current_matrix, self)
        elif algo_name == "A*": path = a_star_search(self.current_matrix, self)
        elif algo_name == "IDA*": path = ida_star_search(self.current_matrix, self)
        elif algo_name == "SHC": path = simple_hill_climbing(self.current_matrix, self)
        elif algo_name == "SAHC": path = steepest_ascent_hill_climbing(self.current_matrix, self)
        elif algo_name == "StocHC": path = stochastic_hill_climbing(self.current_matrix, self)
        elif algo_name == "RRHC": path = random_restart_stochastic_hill_climbing(self.current_matrix, self)
        elif algo_name == "LBS": path = local_beam_search(self.current_matrix, self, k=2)
        elif algo_name == "SA": path = simulated_annealing(self.current_matrix, self, T0=100, Tmin=1, alpha=0.95)
        
        # Xử lý trường hợp bị kẹt cho Local Search
        if path == "LOCAL_MINIMUM":
            error_msg = "Kẹt ở Local Minimum"
            self.update_solution_text(error_msg)
            self.log_text.insert(tk.END, f"\n[THÔNG BÁO] {error_msg}\nRobot không thể dọn sạch bản đồ\n")
            self.log_text.see(tk.END)
        elif path is None or len(path) == 0:
            if count_dust(self.current_matrix) == 0:
                self.update_solution_text("Bản đồ đã sạch sẽ (Clean).")
            else:
                error_msg = "Không tìm thấy solution"
                self.update_solution_text(error_msg)
                self.log_text.insert(tk.END, f"\n[THÔNG BÁO] {error_msg}!\nRobot đã dừng hoạt động.\n")
                self.log_text.see(tk.END)
        else:
            self.update_solution_text(f"Total steps: {len(path)}\nPath: " + " -> ".join(path))
            self.animate_path(path)
            
        self.is_running = False

    def animate_path(self, path):
        self.log_text.insert(tk.END, f"\n[Executing Solution Path]\n{'='*32}\n")
        for idx, action in enumerate(path):
            self.log_text.insert(tk.END, f"Step {idx+1:02d}: {action}\n")
            self.log_text.see(tk.END)
            
            self.current_matrix = result(self.current_matrix, action)
            self.draw_grid(self.current_matrix)
            
            self.root.update()
            time.sleep(0.3)
        
        self.log_text.insert(tk.END, "\nAll dust cleaned.")
        self.log_text.see(tk.END)

    def generate_random_single_state(self, size):
        """Tạo một ma trận ngẫu nhiên làm trạng thái thành phần cho Belief State"""
        matrix = [[0 for _ in range(size)] for _ in range(size)]
        matrix[random.randint(0, size-1)][random.randint(0, size-1)] = 2
        
        # Thêm 2 tường và 2-4 hạt bụi ngẫu nhiên
        for _ in range(2):
            rx, ry = random.randint(0, size-1), random.randint(0, size-1)
            if matrix[rx][ry] == 0: matrix[rx][ry] = 3
        for _ in range(random.randint(2, 4)):
            rx, ry = random.randint(0, size-1), random.randint(0, size-1)
            if matrix[rx][ry] == 0: matrix[rx][ry] = 1
        return matrix

    def generate_belief_states(self):
        """Khởi tạo Belief State với 2 trạng thái ngẫu nhiên"""
        if self.is_running: return
        size = int(self.size_var.get().split('x')[0])
        
        state1 = self.generate_random_single_state(size)
        state2 = self.generate_random_single_state(size)
        self.current_belief_states = [state1, state2]
        
        self.draw_dual_grid(state1, state2)
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, "Generated Belief State (2 Random States).\\nReady for Sensorless Search.\\n")
        self.update_solution_text("")
        
    def generate_partial_belief_states(self):
        """Khởi tạo BS với một số ô biết trước (ràng buộc 2 trạng thái phải giống nhau ở các ô đó)"""
        if self.is_running: return
        size = int(self.size_var.get().split('x')[0])
        
        # 1. Random số lượng ô biết trước (ví dụ 30-40% số ô của bản đồ)
        num_known = max(1, (size * size) // 3)
        
        # 2. Tạo State 1 (Gốc)
        state1 = self.generate_random_single_state(size)
        
        # 3. Chọn ngẫu nhiên các toạ độ làm "ô đã biết"
        all_coords = [(i, j) for i in range(size) for j in range(size)]
        self.known_coords = random.sample(all_coords, num_known)
        
        # 4. Tạo State 2 đảm bảo khớp với State 1 tại các ô đã biết
        while True:
            state2 = [[0 for _ in range(size)] for _ in range(size)]
            robot_in_known = False
            
            # Copy các ô đã biết từ State 1 sang State 2
            for (x, y) in self.known_coords:
                state2[x][y] = state1[x][y]
                if state2[x][y] == 2:
                    robot_in_known = True
                    
            unknown_coords = [c for c in all_coords if c not in self.known_coords]
            
            # Nếu robot chưa nằm trong vùng "đã biết", phải đặt robot vào 1 ô "chưa biết"
            if not robot_in_known:
                if not unknown_coords: continue # Lỗi hiếm: không còn chỗ trống
                rx, ry = random.choice(unknown_coords)
                state2[rx][ry] = 2
                
            # Random thêm tường và bụi vào các ô chưa biết còn trống
            empty_unknown = [(x, y) for (x, y) in unknown_coords if state2[x][y] == 0]
            
            for _ in range(min(2, len(empty_unknown))): # Thêm tối đa 2 tường
                rx, ry = random.choice(empty_unknown)
                state2[rx][ry] = 3
                empty_unknown.remove((rx, ry))
                
            for _ in range(min(random.randint(2, 3), len(empty_unknown))): # Thêm 2-3 bụi
                rx, ry = random.choice(empty_unknown)
                state2[rx][ry] = 1
                empty_unknown.remove((rx, ry))
                
            break
            
        self.current_belief_states = [state1, state2]
        
        # Vẽ lên màn hình với danh sách các ô đã biết (để tạo viền vàng)
        self.draw_dual_grid(state1, state2, self.known_coords)
        
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, f"Generated Partially Observable BS.\nCó {num_known} ô (viền vàng) được nhìn thấy trước.\nReady for Partial Obs Search.\n")
        self.update_solution_text("")

    def draw_dual_grid(self, matrix1, matrix2, known_coords=None):
        """Vẽ 2 ma trận song song. Thêm viền vàng cho các ô đã biết (known_coords)"""
        self.canvas.delete("all")
        size = len(matrix1)
        padding = 20
        grid_width = (self.canvas_dim - padding) / 2
        cell_size = grid_width / size
        
        colors = {0: "#A5D6A7", 1: "#E0E0E0", 2: "#64B5F6", 3: "#E57373"}
        labels = {0: "Clean", 1: "Dust", 2: "Robot", 3: "Wall"}
        font_size = max(6, int(10 - (size - 3) * 1.5))
        
        matrices = [(matrix1, 0), (matrix2, grid_width + padding)]
        
        for matrix, x_offset in matrices:
            for i in range(size):
                for j in range(size):
                    val = matrix[i][j]
                    x1 = x_offset + j * cell_size
                    y1 = i * cell_size
                    x2 = x1 + cell_size
                    y2 = y1 + cell_size
                    
                    # Nếu ô nằm trong danh sách "đã biết", tô viền vàng dày
                    is_known = known_coords and (i, j) in known_coords
                    outline_color = "#FFD700" if is_known else "#FFFFFF"
                    outline_width = 3 if is_known else 1
                    
                    self.canvas.create_rectangle(x1, y1, x2, y2, fill=colors[val], outline=outline_color, width=outline_width)
                    self.canvas.create_text(x1 + cell_size/2, y1 + cell_size/2, 
                                            text=labels[val], font=("Segoe UI", font_size, "bold"), fill="#424242" if val in [0, 1] else "#FFFFFF")
        self.root.update_idletasks()

    def log_bs_step(self, current_node, frontier, closed_list, closed_name):
        """Format log đơn giản: Chỉ in ra chuỗi hành động của Node đang xét và Frontier"""
        
        # 1. Lấy chuỗi hành động dẫn đến node hiện tại
        current_path = solution(current_node)
        current_action_str = " -> ".join(current_path) if current_path else "Start (Chưa di chuyển)"
        
        # 2. Lấy chuỗi hành động của các node đang chờ trong Frontier
        frontier_actions = []
        for n in frontier:
            path = solution(n)
            # Rút gọn chữ cái đầu cho dễ nhìn nếu chuỗi quá dài (U, D, L, R)
            short_path = [self.short_action(act) for act in path]
            act_str = "-".join(short_path) if short_path else "Start"
            frontier_actions.append(f"[{act_str}]")
            
        frontier_display = ", ".join(frontier_actions) if frontier_actions else "[]"

        # 3. Xuất ra màn hình Log
        log_msg = f"Đang xét chuỗi: {current_action_str}\n"
        log_msg += f"Frontier: {frontier_display}\n"
        log_msg += "-" * 40 + "\n"
        
        self.log_text.insert(tk.END, log_msg)
        self.log_text.see(tk.END) # Tự động cuộn xuống dòng mới nhất
        self.root.update()

    def run_sensorless_algo(self):
        """Chạy thuật toán Sensorless"""
        if self.is_running: return
        # Nếu chưa có belief state, tự động tạo
        if not hasattr(self, 'current_belief_states'):
            self.generate_belief_states()
            
        self.is_running = True
        BeliefNode.reset_counter()
        self.explored_bs_names = []
        
        algo_name = "Sensorless Search"
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, f"[{algo_name}] Execution Started\n{'='*40}\n")
        self.update_solution_text("Searching in Belief Space...")
        
        path = sensorless_bfs(self.current_belief_states, self)
        
        if path is None or len(path) == 0:
            if bs_goal_test(self.current_belief_states):
                self.update_solution_text("Cả 2 trạng thái khởi tạo đều đã sạch (Clean).")
            else:
                self.update_solution_text("Không tìm thấy solution")
                self.log_text.insert(tk.END, f"\n[THÔNG BÁO] Kẹt! Không tìm được chuỗi hành động chung.\n")
        else:
            self.update_solution_text(f"Total steps: {len(path)}\nPath: " + " -> ".join(path))
            self.animate_bs_path(path)
            
        self.is_running = False
        
    def run_partial_obs_algo(self):
        """Chạy thuật toán tìm kiếm cho môi trường nhìn thấy một phần"""
        if self.is_running: return
        
        # Nếu chưa có belief state kiểu partial, tự động tạo
        if not hasattr(self, 'current_belief_states') or not hasattr(self, 'known_coords'):
            self.generate_partial_belief_states()
            
        self.is_running = True
        BeliefNode.reset_counter()
        self.explored_bs_names = []
        
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, f"[Partially Observable BFS] Execution Started\n{'='*40}\n")
        self.update_solution_text("Searching with Partial Knowledge...")
        
        path = partial_obs_bfs(self.current_belief_states, self)
        
        if path is None or len(path) == 0:
            if bs_goal_test(self.current_belief_states):
                self.update_solution_text("Cả 2 trạng thái đều đã sạch (Clean).")
            else:
                self.update_solution_text("Không tìm thấy solution")
                self.log_text.insert(tk.END, f"\n[THÔNG BÁO] Kẹt! Không tìm được chuỗi hành động chung.\n")
        else:
            self.update_solution_text(f"Total steps: {len(path)}\nPath: " + " -> ".join(path))
            
            # Chạy hoạt ảnh và tắt viền vàng (truyền None cho draw_dual_grid trong hàm animate)
            self.animate_bs_path(path)
            
        self.is_running = False

    def animate_bs_path(self, path):
        """Chạy hoạt ảnh cho Belief State"""
        self.log_text.insert(tk.END, f"\n[Executing Solution Path]\n{'='*32}\n")
        states = self.current_belief_states
        
        for idx, action in enumerate(path):
            self.log_text.insert(tk.END, f"Step {idx+1:02d}: {action}\n")
            self.log_text.see(tk.END)
            
            # Chỉ áp dụng rule của loại 1 (Đã Goal thì nằm im)
            states = bs_result(states, action)
            
            self.current_belief_states = states
            self.draw_dual_grid(states[0], states[1])
            self.root.update()
            time.sleep(0.5)
            
        self.log_text.insert(tk.END, "\nAll possible states are now clean.")
        self.log_text.see(tk.END)

In [451]:
# Chạy chương trình
if __name__ == "__main__":
    root = tk.Tk()
    app = CleaningRobotGUI(root)
    root.mainloop()